### Prepare dialogue 

In [3]:
import pandas as pd

with open("../../own_script/dialogue_2/dialogue_2.txt", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]

df = pd.DataFrame({
    "utterance_id": range(1, len(lines)+1),
    "text": lines,
})
df.head()

,utterance_id,text
0,1,Why are you bothering me? What's the problem?
1,2,"Ahh that thing again, can you just stay away f..."
2,3,I'm fine! I am very good and doing well at the...
3,4,"Besides, you are the one who seems to be doing..."


### Load text/VAD model

In [4]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)


c:\Users\Legion 5 Pro\.conda\envs\wagner2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.10.0+cpu
transformers: 5.2.0


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "RobroKools/vad-bert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 875.39it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: RobroKools/vad-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

### Predict function

In [6]:
import numpy as np

def predict_vad(texts):
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
    # logits shape: [batch, 3] = [V, A, D]
    vad = out.logits.cpu().numpy()
    return vad  # np.array [batch,3]


### Check value range of vac-bert

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tok = AutoTokenizer.from_pretrained("RobroKools/vad-bert")
model = AutoModelForSequenceClassification.from_pretrained("RobroKools/vad-bert")

samples = [
    "I feel terrible and hopeless.",
    "I feel completely neutral.",
    "I feel amazing and so happy!",
]

for s in samples:
    inputs = tok(s, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs).logits.squeeze().tolist()
    print(s, "-> VAD:", out)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 949.85it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: RobroKools/vad-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


I feel terrible and hopeless. -> VAD: [1.7622524499893188, 3.2708568572998047, 2.4671969413757324]
I feel completely neutral. -> VAD: [2.781001567840576, 2.894531488418579, 3.203058958053589]
I feel amazing and so happy! -> VAD: [4.503335952758789, 4.0434064865112305, 3.532799482345581]


In [8]:
samples = [
    "I want to die. I hate everything.",
    "I feel completely empty and numb.",
    "This is fine.",
    "I'm a little annoyed.",
    "I'm so excited I can't stop screaming!",
    "I feel calm, peaceful, and relaxed.",
]

vals = []
for s in samples:
    inputs = tok(s, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs).logits.squeeze().tolist()  # [V,A,D]
    print(s, "->", out)
    vals.append(out)

import numpy as np
vals = np.array(vals)
print("Valence range:", vals[:,0].min(), vals[:,0].max())
print("Arousal range:", vals[:,1].min(), vals[:,1].max())
print("Dominance range:", vals[:,2].min(), vals[:,2].max())


I want to die. I hate everything. -> [1.4248108863830566, 3.895531415939331, 2.7246110439300537]
I feel completely empty and numb. -> [1.9501692056655884, 3.131157159805298, 2.477699041366577]
This is fine. -> [3.1369645595550537, 3.0026485919952393, 3.1079323291778564]
I'm a little annoyed. -> [1.9209797382354736, 3.553025960922241, 2.9750216007232666]
I'm so excited I can't stop screaming! -> [3.0788471698760986, 4.616365909576416, 3.017502546310425]
I feel calm, peaceful, and relaxed. -> [3.758591413497925, 2.6541788578033447, 3.186662435531616]
Valence range: 1.4248108863830566 3.758591413497925
Arousal range: 2.6541788578033447 4.616365909576416
Dominance range: 2.477699041366577 3.186662435531616


### Test on dialogue 2 and save into dataframe

In [9]:
vad = predict_vad(df["text"].tolist())
df["valence_text"] = vad[:, 0]
df["arousal_text"] = vad[:, 1]
df["dominance_text"] = vad[:, 2]

df.head()

,utterance_id,text,valence_text,arousal_text,dominance_text
0,1,Why are you bothering me? What's the problem?,2.515900,3.496335,3.119745
1,2,"Ahh that thing again, can you just stay away f...",2.503102,3.572586,3.061257
2,3,I'm fine! I am very good and doing well at the...,4.039449,3.855152,3.643599
3,4,"Besides, you are the one who seems to be doing...",3.347648,3.262022,3.264729


### Check with own dataset

In [10]:
import numpy as np

v = df["valence_text"].to_numpy()
a = df["arousal_text"].to_numpy()

print("Valence min/max:", v.min(), v.max())
print("Valence q1/median/q3:", np.quantile(v, [0.25, 0.5, 0.75]))

print("Arousal min/max:", a.min(), a.max())
print("Arousal q1/median/q3:", np.quantile(a, [0.25, 0.5, 0.75]))


Valence min/max: 2.5031023 4.039449
Valence q1/median/q3: [2.51270086 2.93177414 3.52059823]
Arousal min/max: 3.2620218 3.8551524
Arousal q1/median/q3: [3.4377569  3.53446054 3.64322746]


### Scale mismatch issue --> Normalization

- The vac-bert I used has scale mismatch from wagner ([-1, 1]) where as vac-bert scale is ([1, 5])

- To solve this, I implement the normalization method to vac-bert scale to be compatible with wagner and the paper

In [11]:
import numpy as np

# กำหนดช่วง text สมมติเป็น 1..5
V_MIN, V_MAX = 1.0, 5.0

def to_minus1_1(x, xmin=V_MIN, xmax=V_MAX):
    return 2 * (x - xmin) / (xmax - xmin) - 1  # map [xmin,xmax] -> [-1,1]

df["valence_text_n"]  = to_minus1_1(df["valence_text"])
df["arousal_text_n"]  = to_minus1_1(df["arousal_text"])
df["dominance_text_n"] = to_minus1_1(df["dominance_text"])


In [12]:
df

,utterance_id,text,valence_text,arousal_text,dominance_text,valence_text_n,arousal_text_n,dominance_text_n
0,1,Why are you bothering me? What's the problem?,2.515900,3.496335,3.119745,-0.242050,0.248168,0.059873
1,2,"Ahh that thing again, can you just stay away f...",2.503102,3.572586,3.061257,-0.248449,0.286293,0.030629
2,3,I'm fine! I am very good and doing well at the...,4.039449,3.855152,3.643599,0.519725,0.427576,0.321799
3,4,"Besides, you are the one who seems to be doing...",3.347648,3.262022,3.264729,0.173824,0.131011,0.132364


### Assert mutual absolute scale --> Check by revert back to its previous form 

- The vac-bert I used has scale mismatch from wagner ([-1, 1]) where as vac-bert scale is ([1, 5])

- To check this, I implement the revert method to vac-bert scale and assert it to contain the same value for original scale then there is unchanged in absolute meaning in value

- The output must be 0

In [13]:
# map 1..5 -> -1..1
def one5_to_minus1_1(x, xmin=1.0, xmax=5.0):
    x01 = (x - xmin) / (xmax - xmin)
    return 2*x01 - 1

def minus1_1_to_one5(y, xmin=1.0, xmax=5.0):
    x01 = (y + 1) / 2
    return x01 * (xmax - xmin) + xmin

diff = df["arousal_text"] - minus1_1_to_one5(df["arousal_text_n"])
print(diff.abs().max())   # ควร ~ 0 (มีแค่ numerical noise ระดับ 1e-7)


0.0


### Check rank & order value

- The output must be 1

In [14]:
# index ของค่า arousal สูงสุด ก่อนและหลัง normalize ต้องเป็นอันเดียวกัน
orig_argmax = df["arousal_text"].idxmax()
norm_argmax = df["arousal_text_n"].idxmax()
print(orig_argmax, norm_argmax)

# หรือ correlation ระหว่างค่าเดิมกับค่าที่ normalize ควร = 1
df[["arousal_text", "arousal_text_n"]].corr()


2 2


,arousal_text,arousal_text_n
arousal_text,1.0,1.0
arousal_text_n,1.0,1.0


### Check distribution

In [15]:
print(df["arousal_text"].describe())
print(df["arousal_text_n"].describe())


count    4.000000
mean     3.546524
std      0.244534
min      3.262022
25%      3.437757
50%      3.534461
75%      3.643227
max      3.855152
Name: arousal_text, dtype: float64
count    4.000000
mean     0.273262
std      0.122267
min      0.131011
25%      0.218878
50%      0.267230
75%      0.321614
max      0.427576
Name: arousal_text_n, dtype: float64


### Save to .csv format

In [ ]:
df.to_csv("../../own_script/dialogue_2/dialogue_2_vad_text.csv", index=False)